# Presentation figures

**Fig A** — Two-panel: mechanisms (left) × traits (right)  
One rainfall level (~450 mm mean, rf_scale=1.0), N=200 paired seasons per bar.

- Left: isolated contribution of each mechanism (shade, HR, canopy interception) for two phenologies (lf=0.0 vs lf=1.0)
- Right: isolated effect of switching each trait from competitive → facilitative value, all mechanisms on

In [ ]:
import nbformat, sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

# Load class-definition and helper cells from tree.ipynb
_nb = nbformat.read('../tree.ipynb', as_version=4)
for _cell in _nb.cells:
    if _cell.cell_type == 'code' and _cell.get('id', '') in {
        'c23f275b',   # core imports: numpy, farm classes, DataFrame, zeros
        '9adc45cd',   # Tree class
        '2149719d',   # shade_factor()
        '3bed3936',   # TreeCropModel
        '6jrb65spyt', # _m_yield, _b_yield
        'b2d38c76',   # run_one_season, season_metrics, _tree_sf
    }:
        src = _cell['source'].replace("'data/", "'../data/").replace('"data/', '"../data/')
        exec(src, globals())

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

print('Setup complete.')

In [ ]:
def _dY_array(tree_params, climate_obj, seeds, hr_max=1.0, s_deep_0=None, **run_kwargs):
    """Return array of (Y_tree_crop - Y_mono) for each seed.
    Paired simulations: same seed → same rainfall year for both models.
    run_kwargs (shade_on, hr_on, ci_on, deep_roots) forwarded to model.run().
    """
    sf_ = _tree_sf(tree_params)
    lgp = 180
    out = []
    for seed in seeds:
        np.random.seed(seed)
        climate_obj.rainfall = Climate.generate(climate_obj.alpha_r, climate_obj.lambda_r)

        # monoculture
        s_m = Soil(texture='loam'); c_m = Crop(soil=s_m); c_m.lgp = lgp
        s_m.set_nZr(c_m)
        m_m = CropModel(crop=c_m, climate=climate_obj, soil=s_m)
        m_m.run()
        _, _, y_m, _ = season_metrics(c_m, m_m.output(), lgp)

        # tree-crop
        s_t = Soil(texture='loam'); c_t = Crop(soil=s_t); c_t.lgp = lgp
        tr_ = Tree(soil=s_t, **tree_params)
        m_t = TreeCropModel(crop=c_t, tree=tr_, climate=climate_obj, soil=s_t, hr_max=hr_max)
        m_t.run(s_deep_0=s_deep_0, **run_kwargs)
        _, _, y_t, _ = season_metrics(c_t, m_t.output(), lgp, sf=sf_)

        out.append(y_t - y_m)
    return np.array(out)

print('_dY_array ready.')

In [ ]:
N = 200
seeds = np.arange(N)
climate = Climate()   # default: alpha=10 mm, lambda=0.25/day → ~450 mm mean

# ── Mechanisms panel ─────────────────────────────────────────────────────────
# Reference tree: deep-rooted, moderate demand.
# Each bar = isolated effect of ONE mechanism (others off).
_tp_mech = dict(Zr=1500, T_MAX=4.0, kc=1.0, canopy_cover=0.2,
                sw_MPa=-2.0, s_star_MPa=-0.1)

mech_data = {}
for lf in [0.0, 1.0]:
    tp = {**_tp_mech, 'leaf_fraction': lf}
    kw = dict(deep_roots=True)
    dY_base  = np.median(_dY_array(tp, climate, seeds, shade_on=False, hr_on=False, ci_on=False, **kw))
    dY_shade = np.median(_dY_array(tp, climate, seeds, shade_on=True,  hr_on=False, ci_on=False, **kw))
    dY_hr    = np.median(_dY_array(tp, climate, seeds, shade_on=False, hr_on=True,  ci_on=False, **kw))
    dY_ci    = np.median(_dY_array(tp, climate, seeds, shade_on=False, hr_on=False, ci_on=True,  **kw))
    mech_data[lf] = [dY_base, dY_shade, dY_hr, dY_ci]
    print(f'lf={lf}: base={dY_base:.0f}  shade={dY_shade:.0f}  HR={dY_hr:.0f}  CI={dY_ci:.0f}')

# ── Traits panel ─────────────────────────────────────────────────────────────
# Base = "bad" tree (evergreen, shallow, high demand, large canopy). All mechanisms on.
_tp_bad  = dict(Zr=600, T_MAX=4.0, kc=1.0, canopy_cover=0.3,
                sw_MPa=-2.0, s_star_MPa=-0.1, leaf_fraction=1.0)
_kw_all  = dict(shade_on=True, hr_on=True, ci_on=True, deep_roots=True)

dY_bad = np.median(_dY_array(_tp_bad, climate, seeds, **_kw_all))

trait_bars = [
    np.median(_dY_array({**_tp_bad, 'leaf_fraction': 0.0},  climate, seeds, **_kw_all)) - dY_bad,
    np.median(_dY_array({**_tp_bad, 'Zr': 1500},            climate, seeds, **_kw_all)) - dY_bad,
    np.median(_dY_array({**_tp_bad, 'T_MAX': 1.0},          climate, seeds, **_kw_all)) - dY_bad,
    np.median(_dY_array({**_tp_bad, 'canopy_cover': 0.2},   climate, seeds, **_kw_all)) - dY_bad,
]

trait_labels = ['Reverse\nphenology', 'Deep roots', 'Low T_MAX', 'Low canopy\ncover']
print('\nTrait contributions (good − bad):')
for lbl, val in zip(trait_labels, trait_bars):
    print(f'  {lbl.replace(chr(10)," "):25s} {val:+.0f} kg/ha')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fig.subplots_adjust(wspace=0.08)

# ── Left panel: Mechanisms ────────────────────────────────────────────────────
x      = np.arange(4)
width  = 0.35
labels = ['Root\ncompetition', 'Shade', 'HR', 'Canopy\ninterception']
phenol = [(0.0, '#27ae60', 'lf = 0  (dormant)'), (1.0, '#e74c3c', 'lf = 1  (evergreen)')]

for idx, (lf, color, label) in enumerate(phenol):
    offset = (idx - 0.5) * width
    ax1.bar(x + offset, mech_data[lf], width=width * 0.9,
            color=color, alpha=0.85, label=label, zorder=3)

ax1.axhline(0, color='k', lw=1, zorder=2)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=10)
ax1.set_ylabel('ΔY  (tree-crop − mono)  [kg/ha]', fontsize=11)
ax1.set_title('(a)  Mechanisms', fontsize=12, loc='left')
ax1.legend(fontsize=9, frameon=False, loc='lower right')
despine(ax1)

# ── Right panel: Traits ───────────────────────────────────────────────────────
x2     = np.arange(4)
tlbls  = ['Reverse\nphenology', 'Deep roots', 'Low\nT_MAX', 'Low canopy\ncover']
colors = ['#2980b9' if v >= 0 else '#e74c3c' for v in trait_bars]

ax2.bar(x2, trait_bars, color=colors, alpha=0.85, zorder=3)
ax2.axhline(0, color='k', lw=1, zorder=2)
ax2.set_xticks(x2)
ax2.set_xticklabels(tlbls, fontsize=10)
ax2.set_title('(b)  Traits  (good − bad)', fontsize=12, loc='left')
despine(ax2)

plt.savefig('../output/fig_mechanisms_traits.png', dpi=150, bbox_inches='tight')
plt.show()